In [1]:
from tools import *

import netCDF4 as nc
import numpy as np

In [2]:
mesh_num = 9
filename = f'io/mesh_cvt_{mesh_num}.nc'
ds = nc.Dataset(filename)

ncells = ds.dimensions['nCells'].size
nedges = ds.dimensions['nEdges'].size
print(f'ncells = {ncells}')
print(f'nedges = {nedges}')

ds.close()

ncells = 2621442
nedges = 7864320


In [3]:
def get_bfs_order_patch(filename, patch, npatches):
    ds = nc.Dataset(filename)

    cells_on_cell = ds.variables['cellsOnCell']
    patch_cell = ds.variables[f'patch_cell_{npatches}']
    visited = np.zeros(ds.dimensions['nCells'].size, dtype=int)
    
    root_ind = np.where(patch_cell[:] == patch)[0][0]
    
    sorted_cell_inds = []
    queue = [root_ind]
    visited[root_ind] = 1
    while queue:
        cur_ind = queue.pop(0)
        sorted_cell_inds.append(cur_ind)
    
        adjacent_inds = np.array(cells_on_cell[cur_ind][cells_on_cell[cur_ind] != 0]) - 1
        adjacent_inds = adjacent_inds[np.where(patch_cell[adjacent_inds] == patch)]
        for ind in adjacent_inds:
            if not visited[ind]:
                queue.append(ind)
                visited[ind] = 1
            # END if
        # END for
    # END while
    #print(len(sorted_cell_inds))
    ordered_inds = np.array(sorted_cell_inds)
    
    ds.close()
    return ordered_inds
# END get_bfs_order()

In [4]:
### go through each config for the number of patches
### and perform bfs ordering within each patch,
### recording each ordering in the base mesh file

npatches_lists = get_npatches_lists()
npatches_list = npatches_lists[str(mesh_num)]


for npatches in npatches_list:
    ncells_per_patch = ncells // npatches
    
    print(f'labeling patches for {npatches} patches')
    patch_cell = []
    with open(f'io/cvt{mesh_num}.graph.info.part.{npatches}', 'r') as ifile:
        for line in ifile:
            patch_cell.append(int(line[:-1]))
        # END for
    # END with
    patch_cell = np.array(patch_cell, dtype=int)
    
    for i in range(npatches):
        num = np.where(patch_cell == i)[0].size
        if num != ncells_per_patch:
            print(f'ncells in patch {i} = {num}')
        # END if 
    # END for
    
    ds = nc.Dataset(filename, 'a', format='NETCDF4')
    patch_name_cell = f'patch_cell_{npatches}'
    try:
        ds.createVariable(patch_name_cell, 'i', ('nCells'))
        print(f'created variable {patch_name_cell}')
    except:
        print(f'variable {patch_name_cell} already exists')
    # END try
    ds.variables[patch_name_cell][:] = patch_cell
    ds.close()

    print(f'sorting for {npatches} patches')
    # sort each patch
    all_inds = np.zeros([npatches, ncells_per_patch], dtype=int)
    for i in range(npatches):
        print(f'\tsorting cells in patch {i}')
        all_inds[i] = get_bfs_order_patch(filename, i, npatches)
    # END for

    ds = nc.Dataset(filename, 'a', format='NETCDF4')
    patch_order_name_cell = f'order_patch_cell_{npatches}'
    try:
        ds.createVariable(patch_order_name_cell, 'i', ('nCells'))
        print(f'created variable {patch_order_name_cell}')
    except:
        print(f'variable {patch_order_name_cell} already exists')
    # END try
    ds.variables[patch_order_name_cell][all_inds.flatten()] = np.arange(ncells)
    ds.close()
# END for

labeling patches for 2 patches
created variable patch_cell_2
sorting for 2 patches
	sorting cells in patch 0
	sorting cells in patch 1
created variable order_patch_cell_2
labeling patches for 3 patches
created variable patch_cell_3
sorting for 3 patches
	sorting cells in patch 0
	sorting cells in patch 1
	sorting cells in patch 2
created variable order_patch_cell_3
labeling patches for 6 patches
created variable patch_cell_6
sorting for 6 patches
	sorting cells in patch 0
	sorting cells in patch 1
	sorting cells in patch 2
	sorting cells in patch 3
	sorting cells in patch 4
	sorting cells in patch 5
created variable order_patch_cell_6


In [5]:
### record full bfs order in base mesh file

full_bfs_order = get_bfs_order(filename, 'cells')

ds = nc.Dataset(filename, 'a', format='NETCDF4')

bfs_order_name_cell = 'bfs_order_cell'
try:
    ds.createVariable(bfs_order_name_cell, 'i', ('nCells'))
    print(f'created variable {bfs_order_name_cell}')
except:
    print(f'variable {bfs_order_name_cell} already exists')
# END try
ds.variables[bfs_order_name_cell][full_bfs_order] = np.arange(ncells)

ds.close()

created variable bfs_order_cell


In [6]:
### record spiral order in base mesh file

spiral_cell_inds = get_spiral_cell_order(filename)

ds = nc.Dataset(filename, 'a', format='NETCDF4')

spiral_order_name_cell = 'spiral_order_cell'
try:
    ds.createVariable(spiral_order_name_cell, 'i', ('nCells'))
    print(f'created variable {spiral_order_name_cell}')
except:
    print(f'variable {spiral_order_name_cell} already exists')
# END try
ds.variables[spiral_order_name_cell][spiral_cell_inds] = np.arange(ncells)

ds.close()

created variable spiral_order_cell
